# Notebook 5: Robotics Representations & Action Spaces

This notebook covers robotics-specific concepts essential for VLA and robot learning systems:

1. **Rotation Representations** — quaternion, Euler, and 6D rotation
2. **6D to Rotation Matrix (Gram-Schmidt)** — continuous rotation representation
3. **Action Normalization** — Z-score and quantile normalization
4. **Action Space: Loss & Pre/Postprocessing** — building a complete action space
5. **Action Slicing from Trajectories** — extracting proprio and action chunks

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
from scipy.spatial.transform import Rotation as R

torch.manual_seed(42)
np.random.seed(42)

---
## Exercise 5a: Rotation Representations

In robotics, orientations can be represented as:
- **Quaternions** (4D, compact but discontinuous)
- **Euler angles** (3D, intuitive but have gimbal lock)
- **Rotation matrices** (9D / 3x3, overcomplete but smooth)
- **6D representation** (first two columns of rotation matrix, continuous and compact)

The 6D representation (Zhou et al., "On the Continuity of Rotation Representations in Neural Networks") is preferred for learning because it is **continuous** — nearby rotations have nearby representations.

Implement conversions:
1. Quaternion to rotation matrix
2. Rotation matrix to 6D representation (take first two columns, flatten)
3. Euler angles to 6D representation

In [ ]:
def quat_to_rotmat(q: np.ndarray) -> np.ndarray:
    """
    Convert quaternion(s) to rotation matrix/matrices.

    Parameters
    ----------
    q : ndarray of shape (..., 4), scalar-last format [x, y, z, w]

    Returns
    -------
    ndarray of shape (..., 3, 3)
    """
    # TODO: Use scipy.spatial.transform.Rotation to convert
    # R.from_quat(q).as_matrix()
    # Handle batched input: q can be (4,) or (N, 4)
    raise NotImplementedError("Implement quat_to_rotmat")


def rotmat_to_6d(mat: np.ndarray) -> np.ndarray:
    """
    Extract 6D rotation representation from rotation matrix.

    Take the first two columns of the rotation matrix and flatten.

    Parameters
    ----------
    mat : ndarray of shape (..., 3, 3)

    Returns
    -------
    ndarray of shape (..., 6)
    """
    # TODO:
    # 1. Take first two columns: mat[..., :, :2]  -> (..., 3, 2)
    # 2. Reshape to (..., 6)
    raise NotImplementedError("Implement rotmat_to_6d")


def quat_to_6d(q: np.ndarray) -> np.ndarray:
    """
    Convert quaternion to 6D rotation representation.

    Parameters
    ----------
    q : ndarray (..., 4), scalar-last

    Returns
    -------
    ndarray (..., 6)
    """
    # TODO: Compose quat_to_rotmat and rotmat_to_6d
    raise NotImplementedError("Implement quat_to_6d")


def euler_to_6d(angles: np.ndarray, convention: str = "xyz") -> np.ndarray:
    """
    Convert Euler angles to 6D rotation representation.

    Parameters
    ----------
    angles : ndarray (..., 3) in radians
    convention : str, Euler convention (e.g. 'xyz')

    Returns
    -------
    ndarray (..., 6)
    """
    # TODO:
    # 1. Convert Euler to rotation matrix using R.from_euler(convention, angles)
    # 2. Extract 6D representation using rotmat_to_6d
    raise NotImplementedError("Implement euler_to_6d")

In [ ]:
# === Tests for Exercise 5a ===

# Identity quaternion [0, 0, 0, 1] -> identity rotation matrix
q_id = np.array([0.0, 0.0, 0.0, 1.0])
mat_id = quat_to_rotmat(q_id)
assert mat_id.shape == (3, 3), f"Expected (3, 3), got {mat_id.shape}"
assert np.allclose(mat_id, np.eye(3), atol=1e-6), "Identity quat -> identity matrix"

# 6D of identity should be [1,0,0, 0,1,0]
r6d_id = rotmat_to_6d(mat_id)
assert r6d_id.shape == (6,), f"Expected (6,), got {r6d_id.shape}"
assert np.allclose(r6d_id, [1, 0, 0, 0, 1, 0], atol=1e-6), f"Identity 6D: {r6d_id}"

# Batch test
q_batch = np.random.randn(10, 4)
q_batch = q_batch / np.linalg.norm(q_batch, axis=-1, keepdims=True)  # normalize
mat_batch = quat_to_rotmat(q_batch)
assert mat_batch.shape == (10, 3, 3)

r6d_batch = quat_to_6d(q_batch)
assert r6d_batch.shape == (10, 6)

# Euler to 6D
euler_zero = np.zeros(3)
r6d_euler = euler_to_6d(euler_zero)
assert r6d_euler.shape == (6,)
assert np.allclose(r6d_euler, [1, 0, 0, 0, 1, 0], atol=1e-6), "Zero Euler -> identity 6D"

# Batch Euler
euler_batch = np.random.randn(5, 3) * 0.5
r6d_euler_batch = euler_to_6d(euler_batch)
assert r6d_euler_batch.shape == (5, 6)

# Verify rotation matrices are orthonormal
for i in range(10):
    m = mat_batch[i]
    assert np.allclose(m @ m.T, np.eye(3), atol=1e-5), f"R^T R should be I (sample {i})"
    assert np.allclose(np.linalg.det(m), 1.0, atol=1e-5), f"det(R) should be 1 (sample {i})"

print("Exercise 5a passed!")

---
## Exercise 5b: 6D Rotation to Rotation Matrix (Gram-Schmidt)

To recover a valid rotation matrix from a 6D representation, apply Gram-Schmidt orthogonalization:

Given 6D vector $[a_1 \| a_2]$ where $a_1, a_2 \in \mathbb{R}^3$:

$$b_1 = \frac{a_1}{\|a_1\|}$$
$$b_2 = \frac{a_2 - (b_1 \cdot a_2)\, b_1}{\|a_2 - (b_1 \cdot a_2)\, b_1\|}$$
$$b_3 = b_1 \times b_2$$

$$R = [b_1 \mid b_2 \mid b_3]$$

This procedure is differentiable, which is why the 6D representation works well with neural networks.

In [ ]:
def rotation_6d_to_matrix_np(v6: np.ndarray) -> np.ndarray:
    """
    Convert 6D rotation representation to rotation matrix using Gram-Schmidt.

    Parameters
    ----------
    v6 : ndarray (..., 6)

    Returns
    -------
    ndarray (..., 3, 3)
    """
    # TODO:
    # 1. Split v6 into a1 = v6[..., :3] and a2 = v6[..., 3:6]
    # 2. Normalize a1: b1 = a1 / ||a1||
    # 3. Project out b1 from a2: proj = (b1 . a2) * b1, residual = a2 - proj
    # 4. Normalize: b2 = residual / ||residual||
    # 5. Cross product: b3 = b1 x b2
    # 6. Stack columns: R = stack([b1, b2, b3], axis=-1)  -> (..., 3, 3)
    raise NotImplementedError("Implement rotation_6d_to_matrix_np")


def rotation_6d_to_matrix_torch(v6: torch.Tensor) -> torch.Tensor:
    """
    Same as above but in PyTorch (differentiable).

    Parameters
    ----------
    v6 : Tensor (..., 6)

    Returns
    -------
    Tensor (..., 3, 3)
    """
    # TODO: Same steps as numpy version, using torch operations
    # Use torch.linalg.cross for cross product (or torch.cross with dim=-1)
    # Use torch.linalg.norm or manual norm for normalization
    raise NotImplementedError("Implement rotation_6d_to_matrix_torch")

In [ ]:
# === Tests for Exercise 5b ===

# Identity 6D -> identity matrix
v6_id = np.array([1.0, 0, 0, 0, 1, 0])
R_id = rotation_6d_to_matrix_np(v6_id)
assert R_id.shape == (3, 3)
assert np.allclose(R_id, np.eye(3), atol=1e-6), f"Expected I, got\n{R_id}"

# Round-trip: quat -> 6D -> matrix -> check orthonormality
np.random.seed(123)
for _ in range(20):
    q = np.random.randn(4)
    q = q / np.linalg.norm(q)
    r6d = quat_to_6d(q)
    R_recovered = rotation_6d_to_matrix_np(r6d)

    # Orthonormality
    assert np.allclose(R_recovered @ R_recovered.T, np.eye(3), atol=1e-5), "R^T R != I"
    assert np.allclose(np.linalg.det(R_recovered), 1.0, atol=1e-5), "det(R) != 1"

    # Should match original rotation
    R_orig = R.from_quat(q).as_matrix()
    assert np.allclose(R_recovered, R_orig, atol=1e-5), "Recovered matrix should match original"

# Batch numpy
v6_batch = np.random.randn(10, 6)
R_batch = rotation_6d_to_matrix_np(v6_batch)
assert R_batch.shape == (10, 3, 3)
for i in range(10):
    assert np.allclose(R_batch[i] @ R_batch[i].T, np.eye(3), atol=1e-5)

# Torch version
v6_t = torch.randn(5, 6, requires_grad=True)
R_t = rotation_6d_to_matrix_torch(v6_t)
assert R_t.shape == (5, 3, 3)

# Differentiability
loss = R_t.sum()
loss.backward()
assert v6_t.grad is not None, "Must be differentiable"
assert v6_t.grad.shape == (5, 6)

# Torch and numpy should agree
v6_test = np.random.randn(3, 6)
R_np = rotation_6d_to_matrix_np(v6_test)
R_torch = rotation_6d_to_matrix_torch(torch.tensor(v6_test, dtype=torch.float64)).numpy()
assert np.allclose(R_np, R_torch, atol=1e-5), "Numpy and Torch versions should match"

print("Exercise 5b passed!")

---
## Exercise 5c: Action Normalization

Robot actions need to be normalized before training. Two common schemes:

**Z-score normalization:**
$$\hat{x} = \frac{x - \mu}{\sigma + \epsilon}$$
$$x = \hat{x} \cdot (\sigma + \epsilon) + \mu$$

**Quantile normalization** (maps to [-1, 1]):
$$\hat{x} = \frac{x - q_{01}}{q_{99} - q_{01} + \epsilon} \cdot 2 - 1$$
$$x = \frac{\hat{x} + 1}{2} \cdot (q_{99} - q_{01} + \epsilon) + q_{01}$$

In [ ]:
def zscore_normalize(
    x: torch.Tensor,
    mean: torch.Tensor,
    std: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    """
    Z-score normalize: (x - mean) / (std + eps)

    x: (..., D), mean: (D,), std: (D,)
    """
    # TODO: Implement
    raise NotImplementedError


def zscore_unnormalize(
    x_norm: torch.Tensor,
    mean: torch.Tensor,
    std: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    """
    Inverse of Z-score normalize: x_norm * (std + eps) + mean
    """
    # TODO: Implement
    raise NotImplementedError


def quantile_normalize(
    x: torch.Tensor,
    q01: torch.Tensor,
    q99: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    """
    Quantile normalize to [-1, 1]: (x - q01) / (q99 - q01 + eps) * 2 - 1

    x: (..., D), q01: (D,), q99: (D,)
    """
    # TODO: Implement
    raise NotImplementedError


def quantile_unnormalize(
    x_norm: torch.Tensor,
    q01: torch.Tensor,
    q99: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    """
    Inverse: (x_norm + 1) / 2 * (q99 - q01 + eps) + q01
    """
    # TODO: Implement
    raise NotImplementedError

In [ ]:
# === Tests for Exercise 5c ===
torch.manual_seed(0)
D = 7
x = torch.randn(100, D) * 5 + 3  # shifted and scaled data

mean = x.mean(dim=0)
std = x.std(dim=0)
q01 = x.quantile(0.01, dim=0)
q99 = x.quantile(0.99, dim=0)

# Z-score round-trip
x_z = zscore_normalize(x, mean, std)
x_back = zscore_unnormalize(x_z, mean, std)
assert torch.allclose(x, x_back, atol=1e-4), f"Z-score round-trip failed, max diff: {(x - x_back).abs().max()}"

# Z-score normalized data should have ~zero mean and ~unit std
assert x_z.mean(dim=0).abs().max() < 0.1, "Normalized mean should be ~0"
assert (x_z.std(dim=0) - 1.0).abs().max() < 0.1, "Normalized std should be ~1"

# Quantile round-trip
x_q = quantile_normalize(x, q01, q99)
x_back_q = quantile_unnormalize(x_q, q01, q99)
assert torch.allclose(x, x_back_q, atol=1e-4), f"Quantile round-trip failed, max diff: {(x - x_back_q).abs().max()}"

# Quantile normalized: most values in [-1, 1]
in_range = ((x_q >= -1.2) & (x_q <= 1.2)).float().mean()
assert in_range > 0.95, f"Most quantile-normalized values should be near [-1,1], got {in_range:.2%} in range"

# Batch dimensions should work
x_batch = torch.randn(4, 10, D) * 5 + 3
x_z_batch = zscore_normalize(x_batch, mean, std)
assert x_z_batch.shape == (4, 10, D)
x_back_batch = zscore_unnormalize(x_z_batch, mean, std)
assert torch.allclose(x_batch, x_back_batch, atol=1e-4)

print("Exercise 5c passed!")

---
## Exercise 5d: Action Space — Loss and Pre/Postprocessing

Implement a complete action space class similar to `LiberoJointActionSpace` in the SimVLA codebase:

- `dim_action = 7` (delta_xyz(3), delta_euler(3), gripper(1))
- `dim_proprio = 8` (ee_pos(3), ee_ori(3), gripper_states(2))
- `preprocess`: normalize proprio and action
- `compute_loss`: MSE loss between predicted and target velocities
- `postprocess`: unnormalize predicted actions

In [ ]:
class SimpleActionSpace:
    """
    A simple action space with normalization and MSE loss.

    Parameters
    ----------
    dim_action : int
    dim_proprio : int
    action_mean, action_std : Tensor (dim_action,)
    state_mean, state_std : Tensor (dim_proprio,)
    """

    def __init__(
        self,
        dim_action: int = 7,
        dim_proprio: int = 8,
        action_mean: torch.Tensor = None,
        action_std: torch.Tensor = None,
        state_mean: torch.Tensor = None,
        state_std: torch.Tensor = None,
    ):
        self.dim_action = dim_action
        self.dim_proprio = dim_proprio
        self.action_mean = action_mean if action_mean is not None else torch.zeros(dim_action)
        self.action_std = action_std if action_std is not None else torch.ones(dim_action)
        self.state_mean = state_mean if state_mean is not None else torch.zeros(dim_proprio)
        self.state_std = state_std if state_std is not None else torch.ones(dim_proprio)

    def preprocess(
        self,
        proprio: torch.Tensor,   # [B, dim_proprio]
        action: torch.Tensor,    # [B, T, dim_action]
    ) -> tuple:
        """
        Normalize proprio and action using Z-score normalization.

        Returns
        -------
        (proprio_norm, action_norm)
        """
        # TODO:
        # proprio_norm = (proprio - self.state_mean) / (self.state_std + 1e-6)
        # action_norm = (action - self.action_mean) / (self.action_std + 1e-6)
        raise NotImplementedError("Implement preprocess")

    def compute_loss(
        self,
        pred: torch.Tensor,    # [B, T, dim_action] predicted velocity
        target: torch.Tensor,  # [B, T, dim_action] target velocity
    ) -> dict:
        """
        Compute MSE loss.

        Returns
        -------
        dict with 'velocity_loss' key
        """
        # TODO:
        # loss = mean((pred - target)^2)
        # return {"velocity_loss": loss}
        raise NotImplementedError("Implement compute_loss")

    def postprocess(self, action_norm: torch.Tensor) -> torch.Tensor:
        """
        Unnormalize action.

        Returns
        -------
        Tensor [B, T, dim_action]
        """
        # TODO:
        # return action_norm * (self.action_std + 1e-6) + self.action_mean
        raise NotImplementedError("Implement postprocess")

In [ ]:
# === Tests for Exercise 5d ===
torch.manual_seed(50)
dim_a, dim_p = 7, 8
a_mean = torch.randn(dim_a)
a_std = torch.rand(dim_a) + 0.5
s_mean = torch.randn(dim_p)
s_std = torch.rand(dim_p) + 0.5

space = SimpleActionSpace(
    dim_action=dim_a, dim_proprio=dim_p,
    action_mean=a_mean, action_std=a_std,
    state_mean=s_mean, state_std=s_std,
)

# Test preprocess
B, T = 4, 10
proprio = torch.randn(B, dim_p)
action = torch.randn(B, T, dim_a)
p_norm, a_norm = space.preprocess(proprio, action)

assert p_norm.shape == (B, dim_p)
assert a_norm.shape == (B, T, dim_a)

# Test postprocess round-trip
action_recovered = space.postprocess(a_norm)
assert torch.allclose(action, action_recovered, atol=1e-4), "Preprocess + postprocess should be identity"

# Test loss
pred = torch.randn(B, T, dim_a)
target = torch.randn(B, T, dim_a)
loss_dict = space.compute_loss(pred, target)
assert 'velocity_loss' in loss_dict, "Must return dict with 'velocity_loss'"
assert loss_dict['velocity_loss'].ndim == 0, "Loss should be scalar"
assert loss_dict['velocity_loss'].item() > 0, "Loss should be positive for random inputs"

# Zero loss when pred == target
loss_zero = space.compute_loss(target, target)
assert loss_zero['velocity_loss'].item() < 1e-10, "Loss should be 0 when pred == target"

# Default normalization (no stats) should be identity
space_default = SimpleActionSpace()
p_d, a_d = space_default.preprocess(proprio, action)
assert torch.allclose(p_d, proprio, atol=1e-6)
assert torch.allclose(a_d, action, atol=1e-6)

print("Exercise 5d passed!")

---
## Exercise 5e: Action Slicing from Trajectories

Given an absolute trajectory tensor of shape `[H+1, D]` where:
- Row 0 is the current state (proprioception)
- Rows 1..H are future states (used as action targets)

Implement `action_slice` that:
1. Extracts `proprio = trajectory[0]`
2. Extracts `action = trajectory[1:]`
3. Optionally converts specified dimensions from absolute to **delta** (relative to current state):
   `action[:, idx] -= proprio[idx]`

In [ ]:
def action_slice(
    abs_traj: torch.Tensor,            # [H+1, D]
    idx_for_delta: list = (),           # indices to convert to delta
) -> dict:
    """
    Slice a trajectory into proprio and action.

    Parameters
    ----------
    abs_traj : Tensor [H+1, D] — H+1 timesteps, D dimensions
    idx_for_delta : list of int — dimensions to convert to deltas

    Returns
    -------
    dict with:
        'proprio': Tensor [D]
        'action': Tensor [H, D]
    """
    if not isinstance(abs_traj, torch.Tensor):
        raise TypeError("abs_traj must be a torch.Tensor")
    if abs_traj.ndim != 2 or abs_traj.size(0) < 2:
        raise ValueError("abs_traj must be [H+1, D] with H>=1")

    # TODO:
    # 1. proprio = abs_traj[0]           -> [D]
    # 2. action = abs_traj[1:].clone()   -> [H, D]
    # 3. If idx_for_delta is non-empty:
    #      Convert idx_for_delta to a LongTensor
    #      action[:, idx] -= proprio[idx]
    # 4. Return {"proprio": proprio, "action": action}
    raise NotImplementedError("Implement action_slice")

In [ ]:
# === Tests for Exercise 5e ===

# Basic test
traj = torch.tensor([
    [1.0, 2.0, 3.0],  # proprio (t=0)
    [1.5, 2.5, 3.5],  # action (t=1)
    [2.0, 3.0, 4.0],  # action (t=2)
])
result = action_slice(traj)
assert torch.allclose(result['proprio'], torch.tensor([1.0, 2.0, 3.0]))
assert result['action'].shape == (2, 3)
assert torch.allclose(result['action'][0], torch.tensor([1.5, 2.5, 3.5]))
assert torch.allclose(result['action'][1], torch.tensor([2.0, 3.0, 4.0]))

# With delta conversion for first two dims
result_delta = action_slice(traj, idx_for_delta=[0, 1])
assert torch.allclose(result_delta['proprio'], torch.tensor([1.0, 2.0, 3.0]))
# action[0] should be [1.5-1.0, 2.5-2.0, 3.5] = [0.5, 0.5, 3.5]
assert torch.allclose(result_delta['action'][0], torch.tensor([0.5, 0.5, 3.5]))
# action[1] should be [2.0-1.0, 3.0-2.0, 4.0] = [1.0, 1.0, 4.0]
assert torch.allclose(result_delta['action'][1], torch.tensor([1.0, 1.0, 4.0]))

# Without delta, original action should be preserved
result_no_delta = action_slice(traj, idx_for_delta=[])
assert torch.allclose(result_no_delta['action'][0], torch.tensor([1.5, 2.5, 3.5]))

# Error handling
try:
    action_slice(torch.tensor([1.0, 2.0]))  # 1D tensor
    assert False, "Should raise ValueError for 1D input"
except ValueError:
    pass

try:
    action_slice("not a tensor")
    assert False, "Should raise TypeError for non-tensor"
except TypeError:
    pass

# Larger trajectory
H, D = 30, 8
big_traj = torch.randn(H + 1, D)
result_big = action_slice(big_traj, idx_for_delta=[0, 1, 2])
assert result_big['proprio'].shape == (D,)
assert result_big['action'].shape == (H, D)

# Verify delta computation for large traj
for i in range(3):
    expected_delta = big_traj[1:, i] - big_traj[0, i]
    assert torch.allclose(result_big['action'][:, i], expected_delta, atol=1e-6)

# Non-delta dims should be unchanged
for i in range(3, D):
    assert torch.allclose(result_big['action'][:, i], big_traj[1:, i], atol=1e-6)

print("Exercise 5e passed!")
print("\n=== All Notebook 5 exercises passed! ===")